# 06 · Evaluation and comparison

**Input:** `val/test.pkl`, `outputs/predictions/` (notebooks 04, 05), models in `outputs/`. **Output:** `outputs/results_test.csv`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

In [ ]:
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display
from sklearn.metrics import confusion_matrix
from transformers import AutoModelForSequenceClassification, AutoTokenizer

from src.config import BERT_DIR, BILSTM_DIR, CFG, DEVICE, LABELS, ON_GPU, OUTPUT_DIR, PALETTE, setup_notebook
from src.data import Vocab, load_splits
from src.evaluation import (METRIC_COLS, bootstrap_macro_f1, compute_metrics, confusion_table, load_predictions,
                            paired_difference, plot_confusion, predict_lstm_probs, predict_transformer,
                            predict_with_neutral_bias, summarize_bootstrap, tune_neutral_bias)
from src.models import BiLSTMAttention

setup_notebook()
_, val_df, test_df = load_splits()
y_val, y_test = val_df['label'].values, test_df['label'].values
print(f'test: {len(test_df):,} review')

## 1. Load predictions

In [ ]:
PRED = {k: v for k, v in load_predictions().items() if k in ('BERT fine-tune', 'BiLSTM + Attention')}
PRED['Always positive'] = {'probs': np.tile([0., 0., 1.], (len(test_df), 1)), 'test_index': None,
                              'train_time_s': 0, 'infer_time_s': 0, 'n_params': 0}

full_models = list(PRED)
for name, p in PRED.items():
    p['pred'] = p['probs'].argmax(1)
    print(f'{name:22s} predictions on the full test set')

## 2. Results on the full test set

In [ ]:
rows = {}
for name in full_models:
    p = PRED[name]
    rows[name] = {**compute_metrics(y_test, p['pred']),
                  'Train (s)': p['train_time_s'], 'Inference (s)': p['infer_time_s'], 'Parameters': p['n_params']}
results_df = pd.DataFrame(rows).T.sort_values('Macro-F1', ascending=False)
display(results_df.style
        .format({**{c: '{:.4f}' for c in METRIC_COLS}, 'Train (s)': '{:,.0f}', 'Inference (s)': '{:,.1f}', 'Parameters': '{:,.0f}'}, na_rep='–')
        .highlight_max(subset=METRIC_COLS, color='#c7e9c0')
        .set_caption(f'Test set results (n = {len(y_test):,})'))
results_df.to_csv(OUTPUT_DIR / 'results_test.csv')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4.8))
order = results_df['Macro-F1'].sort_values().index
axes[0].barh(order, results_df.loc[order, 'Macro-F1'], color='#4c72b0', label='Macro-F1')
axes[0].barh(order, results_df.loc[order, 'Accuracy'], color='none', edgecolor='#dd8452', lw=2, label='Accuracy')
for i, name in enumerate(order):
    axes[0].text(results_df.loc[name, 'Macro-F1'] - 0.01, i, f'{results_df.loc[name, "Macro-F1"]:.3f}', va='center', ha='right', color='white', fontsize=10)
axes[0].set(xlim=(0, 1), title='Macro-F1 (bars) and Accuracy (outline) on the test set')
axes[0].legend(loc='lower right')

per_class = results_df.loc[order[::-1], ['F1 negative', 'F1 neutral', 'F1 positive']]
per_class.columns = LABELS
per_class.plot(kind='bar', ax=axes[1], color=[PALETTE[l] for l in LABELS], rot=0, width=0.8)
axes[1].set(title='F1 per class', ylim=(0, 1))
axes[1].tick_params(axis='x', labelsize=9)
plt.tight_layout()
plt.show()

In [ ]:
dl_models = [n for n in full_models if n != 'Always positive']
fig, axes = plt.subplots(1, len(dl_models), figsize=(5.5 * len(dl_models), 4))
for ax, name in zip(np.atleast_1d(axes), dl_models):
    plot_confusion(y_test, PRED[name]['pred'], name, ax=ax)
plt.tight_layout()
plt.show()

In [ ]:
for name in dl_models:
    display(confusion_table(y_test, PRED[name]['pred']).set_caption(f'{name}: row-normalised confusion matrix (test)'))
for name in dl_models:
    cm = confusion_matrix(y_test, PRED[name]['pred'], labels=[0, 1, 2])
    recall_row, precision_col = cm[1] / cm[1].sum(), cm[:, 1] / cm[:, 1].sum()
    print(f'{name}:')
    print(f'  True neutral reviews are predicted: correct {recall_row[1]:.1%} | as negative {recall_row[0]:.1%} | as positive {recall_row[2]:.1%}')
    print(f'  Reviews predicted neutral are actually: neutral {precision_col[1]:.1%} | negative {precision_col[0]:.1%} | positive {precision_col[2]:.1%}')

## 3. Performance by review length

In [ ]:
from sklearn.metrics import f1_score

bins = [0, 30, 60, 100, 200, 400, np.inf]
bucket_names = ['≤30', '31–60', '61–100', '101–200', '201–400', '>400']
test_df['len_bucket'] = pd.cut(test_df['n_words'], bins=bins, labels=bucket_names)
bucket_masks = [(test_df['len_bucket'] == b).values for b in bucket_names]
by_len = pd.DataFrame({name: [f1_score(y_test[m], PRED[name]['pred'][m], average='macro') if m.any() else np.nan
                              for m in bucket_masks] for name in dl_models}, index=bucket_names)
counts = test_df['len_bucket'].value_counts().reindex(bucket_names)

fig, ax = plt.subplots(figsize=(12, 4.5))
for name in by_len.columns:
    ax.plot(bucket_names, by_len[name], 'o-', label=name)
ax.set(title='Macro-F1 by review length (words) on the test set', xlabel='Words', ylabel='Macro-F1', ylim=(0, 1))
ax.set_xticks(range(len(bucket_names)))
ax.set_xticklabels([f'{b}\n(n={c:,})' for b, c in zip(bucket_names, counts)])
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
display(by_len.round(3).style.set_caption('Macro-F1 by length group (words)'))
for name in by_len.columns:
    valid = by_len[name].dropna()
    print(f'{name}: best in group "{valid.idxmax()}" ({valid.max():.3f}), worst in group "{valid.idxmin()}" ({valid.min():.3f})')
print('Reviews per group:', counts.to_dict())

## 4. Error analysis of the best model

In [ ]:
best_name = max(dl_models, key=lambda n: results_df.loc[n, 'Macro-F1'])
print('Best model:', best_name)
err = test_df[['text', 'Score', 'label']].copy()
err['pred'] = PRED[best_name]['pred']
err['confidence'] = PRED[best_name]['probs'].max(1)

def show_errors(mask, title, k=3):
    rows = err[mask].sort_values('confidence', ascending=False).head(k)
    print(f'\n### {title}  ({mask.sum():,} cases)')
    for _, r in rows.iterrows():
        print(f'- [{r.Score}★ | actual: {LABELS[r.label]} | predicted: {LABELS[r.pred]} ({r.confidence:.0%})] {r.text[:350]}')

show_errors((err['Score'] == 5) & (err['pred'] == 0), '5-star reviews predicted NEGATIVE')
show_errors((err['Score'] == 1) & (err['pred'] == 2), '1-star reviews predicted POSITIVE')
show_errors((err['label'] == 1) & (err['pred'] != 1), '3-star (neutral) reviews misclassified')

In [ ]:
def sample_errors(mask, title, k=15, seed=0):
    rows = err[mask].sample(min(k, int(mask.sum())), random_state=seed)
    print(f'\n### {title}  ({mask.sum():,} cases in total, showing {len(rows)} random cases)')
    for _, r in rows.iterrows():
        print(f'- [{r.Score}★ | predicted: {LABELS[r.pred]} ({r.confidence:.0%})] {r.text[:260]}')

sample_errors((err['Score'] == 5) & (err['pred'] == 0), '5-star reviews predicted NEGATIVE')
sample_errors((err['Score'] == 1) & (err['pred'] == 2), '1-star reviews predicted POSITIVE')

## 5. Bootstrap confidence intervals for Macro-F1

In [ ]:
point = {n: PRED[n]['pred'] for n in full_models}
boot = bootstrap_macro_f1(y_test, point, n_boot=1000)
ci = summarize_bootstrap(boot, {n: results_df.loc[n, 'Macro-F1'] for n in full_models})
display(ci.style.format('{:.4f}').set_caption('Macro-F1 on the full test set, 95% interval (bootstrap)'))
d = paired_difference(boot, 'BERT fine-tune', 'BiLSTM + Attention')
print(f"BERT − BiLSTM: {d['mean']:+.4f}, 95% interval [{d['low']:+.4f}; {d['high']:+.4f}], "
      f"BERT is ahead in {d['P(a > b)']:.1%} of the bootstrap resamples")

## 6. Neutral-threshold calibration

In [ ]:
vocab = Vocab.load(BILSTM_DIR / 'vocab.json')
bilstm = BiLSTMAttention(len(vocab), CFG['lstm_emb_dim'], CFG['lstm_hidden'], CFG['lstm_layers'], 3,
                         CFG['lstm_dropout']).to(DEVICE)
bilstm.load_state_dict(torch.load(BILSTM_DIR / 'model.pt', map_location=DEVICE))
bert_tok = AutoTokenizer.from_pretrained(BERT_DIR)
bert = AutoModelForSequenceClassification.from_pretrained(BERT_DIR).to(DEVICE)
BERT_MAX_LEN = json.loads((BERT_DIR / 'train_info.json').read_text()).get('max_len', 256)

val_probs = {
    'BiLSTM + Attention': predict_lstm_probs(bilstm, vocab, val_df['text'].values, CFG['lstm_max_len']),
    'BERT fine-tune': predict_transformer(bert, bert_tok, val_df['text'].values, BERT_MAX_LEN,
                                                batch_size=128 if ON_GPU else 64, desc='BERT validation'),
}

In [ ]:
rows, curves = [], {}
for name, vp in val_probs.items():
    best_bias, grid, val_scores = tune_neutral_bias(vp, y_val)
    curves[name] = (grid, val_scores)
    test_probs_m = PRED[name]['probs']
    before = compute_metrics(y_test, test_probs_m.argmax(1))
    after = compute_metrics(y_test, predict_with_neutral_bias(test_probs_m, best_bias))
    rows.append({'Model': name, 'bias chosen on validation': best_bias,
                 'Validation Macro-F1 (bias = 0)': val_scores[np.isclose(grid, 0.0)][0],
                 'Validation Macro-F1 (chosen bias)': val_scores.max(),
                 'Test Macro-F1 (before)': before['Macro-F1'], 'Test Macro-F1 (after)': after['Macro-F1'],
                 'Test F1 neutral (before)': before['F1 neutral'], 'Test F1 neutral (after)': after['F1 neutral']})
display(pd.DataFrame(rows).set_index('Model').style.format('{:.4f}'))

fig, ax = plt.subplots(figsize=(8, 4))
for name, (grid, sc) in curves.items():
    ax.plot(grid, sc, label=name)
    ax.axvline(grid[sc.argmax()], ls=':', color='gray')
ax.set(title='Validation Macro-F1 vs. neutral bias', xlabel='bias (added to the neutral log-probability)', ylabel='Validation Macro-F1')
ax.legend()
plt.tight_layout()
plt.show()

## 7. Summary table

Test-set results (80/10/10 split, seed 42; Tesla T4 GPU):

| Model | Accuracy | Macro-F1 | F1 neg | F1 neu | F1 pos | Train time | Inference | Parameters |
|---|---|---|---|---|---|---|---|---|
| **BERT-base** | **0.9074** | **0.7877** | 0.8470 | 0.5540 | 0.9620 | 64.4 min | 83.7 s | 109.48 M |
| BiLSTM + Attention | 0.8788 | 0.7422 | 0.7863 | 0.4916 | 0.9488 | 3.4 min | 3.7 s | 6.25 M |
| Always positive | 0.7794 | 0.2920 | 0 | 0 | 0.8760 | - | - | - |